# CSCI 544 Group 17 - Step 3.2: Next Steps Execution

This notebook implements the four Step 3 next steps without rerunning full 5-fold experiments:

1. Threshold tuning on validation predictions
2. Soft-voting ensemble (best baseline LR + BERTweet probabilities)
3. Class weight adjustment to improve minority-class recall
4. Deeper error analysis comparing transformer failures vs baseline failures

All outputs are saved to:

`step3_2_next_steps_artifacts/`


## Runtime Notes

- This notebook assumes `data/train_processed.csv` and `data/test_processed.csv` already exist from Step 1/2.
- It uses a **single stratified train/validation split** for fast iteration (not full 5-fold rerun).
- For transformer parts, GPU is strongly recommended.


In [ ]:
# Optional Colab install cell (uncomment if needed)
# !pip install -q transformers datasets sentencepiece emoji==0.6.0

import gc
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup


In [ ]:
SEED = 42
VAL_SIZE = 0.2

MODEL_NAME = "vinai/bertweet-base"
EPOCHS = 3
BATCH_SIZE = 16
LR_TRANSFORMER = 2e-5
MAX_LEN = 128


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/CS544-Group17-Project/'
INPUT_DIR = globals().get('INPUT_DIR', PROJECT_DIR + 'input/')
DATA_DIR = globals().get('DATA_DIR', PROJECT_DIR + 'data/')
NEW_OUTPUT_DIR = PROJECT_DIR + 'step3_2_next_steps_artifacts/'

os.makedirs(NEW_OUTPUT_DIR, exist_ok=True)

PROJECT_DIR = Path(PROJECT_DIR)
INPUT_DIR = Path(INPUT_DIR)
DATA_DIR = Path(DATA_DIR)
ARTIFACT_DIR = Path(NEW_OUTPUT_DIR)

print('PROJECT_DIR =', PROJECT_DIR)
print('INPUT_DIR =', INPUT_DIR)
print('DATA_DIR =', DATA_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR = /content/drive/MyDrive/CS544-Group17-Project
INPUT_DIR = /content/drive/MyDrive/CS544-Group17-Project/input
DATA_DIR = /content/drive/MyDrive/CS544-Group17-Project/data
ARTIFACT_DIR = /content/drive/MyDrive/CS544-Group17-Project/step3_2_next_steps_artifacts


In [ ]:
ls /content/drive/MyDrive/CS544-Group17-Project/data/

test_processed.csv  train_processed.csv


In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train_processed.csv')
test_df = pd.read_csv(DATA_DIR / 'test_processed.csv')

if 'keyword_clean' not in train_df.columns:
    train_df['keyword_clean'] = train_df['keyword'].fillna('').astype(str).str.replace('%20', ' ', regex=False)
if 'keyword_clean' not in test_df.columns:
    test_df['keyword_clean'] = test_df['keyword'].fillna('').astype(str).str.replace('%20', ' ', regex=False)

if 'text_clean' not in train_df.columns:
    train_df['text_clean'] = train_df['text'].fillna('').astype(str)
if 'text_clean' not in test_df.columns:
    test_df['text_clean'] = test_df['text'].fillna('').astype(str)

if 'text_transformer' not in train_df.columns:
    train_df['text_transformer'] = train_df['text'].fillna('').astype(str)
if 'text_transformer' not in test_df.columns:
    test_df['text_transformer'] = test_df['text'].fillna('').astype(str)

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('train target distribution:')
print(train_df['target'].value_counts(normalize=True).sort_index())


train shape: (7503, 12)
test shape: (3263, 9)
train target distribution:
target
0    0.574703
1    0.425297
Name: proportion, dtype: float64


In [ ]:
# Shared helpers

def metrics_dict(y_true, y_pred):
    return {
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'Accuracy': accuracy_score(y_true, y_pred),
    }


def threshold_sweep(y_true, probs, start=0.05, end=0.95, step=0.01):
    rows = []
    thresholds = np.round(np.arange(start, end + 1e-9, step), 2)
    for th in thresholds:
        pred = (probs >= th).astype(int)
        m = metrics_dict(y_true, pred)
        rows.append({'threshold': float(th), **m})
    curve = pd.DataFrame(rows).sort_values(['F1', 'Recall', 'Precision'], ascending=False).reset_index(drop=True)
    best = curve.iloc[0].to_dict()
    return curve, best


def save_json(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2)


In [ ]:
# Fixed split for all next-step comparisons
idx = np.arange(len(train_df))
y_all = train_df['target'].astype(int).values

train_idx, val_idx = train_test_split(
    idx,
    test_size=VAL_SIZE,
    stratify=y_all,
    random_state=SEED,
)

tr = train_df.iloc[train_idx].reset_index(drop=True)
va = train_df.iloc[val_idx].reset_index(drop=True)

y_tr = tr['target'].astype(int).values
y_va = va['target'].astype(int).values

print('train split:', tr.shape, 'val split:', va.shape)
print('val target distribution:', pd.Series(y_va).value_counts(normalize=True).sort_index().to_dict())


train split: (6002, 12) val split: (1501, 12)
val target distribution: {0: 0.5749500333111259, 1: 0.42504996668887407}


In [ ]:
# 1) Baseline LR + threshold tuning
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, solver='liblinear', random_state=SEED)),
])

x_tr_tfidf = tr['text_clean'].fillna('').astype(str).values
x_va_tfidf = va['text_clean'].fillna('').astype(str).values
x_test_tfidf = test_df['text_clean'].fillna('').astype(str).values

lr_pipe.fit(x_tr_tfidf, y_tr)

va_prob_lr = lr_pipe.predict_proba(x_va_tfidf)[:, 1]
test_prob_lr = lr_pipe.predict_proba(x_test_tfidf)[:, 1]

lr_curve, lr_best = threshold_sweep(y_va, va_prob_lr)
lr_curve.to_csv(ARTIFACT_DIR / 'baseline_threshold_curve.csv', index=False)

va_pred_lr_default = (va_prob_lr >= 0.5).astype(int)
va_pred_lr_tuned = (va_prob_lr >= lr_best['threshold']).astype(int)

lr_summary = {
    'default_threshold': {k: float(v) for k, v in metrics_dict(y_va, va_pred_lr_default).items()},
    'best_threshold': float(lr_best['threshold']),
    'tuned_threshold': {k: float(lr_best[k]) for k in ['F1', 'Precision', 'Recall', 'Accuracy']},
}

save_json(lr_summary, ARTIFACT_DIR / 'baseline_threshold_summary.json')

print('LR best threshold:', lr_best)


LR best threshold: {'threshold': 0.43, 'F1': 0.7541766109785203, 'Precision': 0.7657512116316639, 'Recall': 0.7429467084639498, 'Accuracy': 0.7941372418387741}


In [ ]:
# Transformer helpers
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = [str(x) for x in texts]
        self.labels = np.array(labels, dtype=np.int64)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }


@torch.no_grad()
def predict_probs(model, dataloader):
    model.eval()
    all_probs = []
    all_labels = []
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels)

    return np.concatenate(all_probs), np.concatenate(all_labels)


def train_transformer_once(train_texts, train_labels, val_texts, val_labels, test_texts, class_weights=None):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

    ds_tr = TweetDataset(train_texts, train_labels, tokenizer, max_len=MAX_LEN)
    ds_va = TweetDataset(val_texts, val_labels, tokenizer, max_len=MAX_LEN)
    ds_te = TweetDataset(test_texts, np.zeros(len(test_texts), dtype=np.int64), tokenizer, max_len=MAX_LEN)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False)
    dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_TRANSFORMER, weight_decay=0.01)
    total_steps = len(dl_tr) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    if class_weights is None:
        criterion = torch.nn.CrossEntropyLoss()
    else:
        w = torch.tensor(class_weights, dtype=torch.float, device=device)
        criterion = torch.nn.CrossEntropyLoss(weight=w)

    history = []
    best_f1 = -1.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for batch in dl_tr:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        avg_loss = running_loss / max(1, len(dl_tr))

        va_prob, va_true = predict_probs(model, dl_va)
        va_pred = (va_prob >= 0.5).astype(int)
        m = metrics_dict(va_true, va_pred)

        row = {
            'epoch': epoch,
            'loss': float(avg_loss),
            'F1': float(m['F1']),
            'Precision': float(m['Precision']),
            'Recall': float(m['Recall']),
            'Accuracy': float(m['Accuracy']),
        }
        history.append(row)
        print(row)

        if m['F1'] > best_f1:
            best_f1 = m['F1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)

    va_prob, va_true = predict_probs(model, dl_va)
    te_prob, _ = predict_probs(model, dl_te)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return va_prob, va_true, te_prob, pd.DataFrame(history)


In [ ]:
# 2) Transformer default + threshold tuning
x_tr_bert = tr['text_transformer'].fillna('').astype(str).values
x_va_bert = va['text_transformer'].fillna('').astype(str).values
x_te_bert = test_df['text_transformer'].fillna('').astype(str).values

va_prob_bert, va_true_bert, test_prob_bert, hist_default = train_transformer_once(
    x_tr_bert,
    y_tr,
    x_va_bert,
    y_va,
    x_te_bert,
    class_weights=None,
)

hist_default.to_csv(ARTIFACT_DIR / 'transformer_default_history.csv', index=False)

bert_curve, bert_best = threshold_sweep(va_true_bert, va_prob_bert)
bert_curve.to_csv(ARTIFACT_DIR / 'transformer_default_threshold_curve.csv', index=False)

bert_summary = {
    'best_threshold': float(bert_best['threshold']),
    'tuned_threshold': {k: float(bert_best[k]) for k in ['F1', 'Precision', 'Recall', 'Accuracy']},
}
save_json(bert_summary, ARTIFACT_DIR / 'transformer_default_threshold_summary.json')

print('BERTweet default best threshold:', bert_best)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

{'epoch': 1, 'loss': 0.4931430447291821, 'F1': 0.7773851590106007, 'Precision': 0.8906882591093117, 'Recall': 0.6896551724137931, 'Accuracy': 0.832111925383078}
{'epoch': 2, 'loss': 0.3502051425265505, 'F1': 0.8006617038875103, 'Precision': 0.8476357267950964, 'Recall': 0.7586206896551724, 'Accuracy': 0.8394403730846103}
{'epoch': 3, 'loss': 0.27758880010112486, 'F1': 0.8077544426494345, 'Precision': 0.8333333333333334, 'Recall': 0.7836990595611285, 'Accuracy': 0.8414390406395736}
BERTweet default best threshold: {'threshold': 0.41, 'F1': 0.8095617529880478, 'Precision': 0.8233387358184765, 'Recall': 0.7962382445141066, 'Accuracy': 0.8407728181212525}


In [ ]:
# 3) Soft-voting ensemble search (LR + BERTweet)
rows = []
for w_bert in np.round(np.arange(0.0, 1.0 + 1e-9, 0.05), 2):
    w_lr = 1.0 - w_bert
    va_prob_ens = w_bert * va_prob_bert + w_lr * va_prob_lr
    curve, best = threshold_sweep(y_va, va_prob_ens)
    rows.append({
        'w_bert': float(w_bert),
        'w_lr': float(w_lr),
        'best_threshold': float(best['threshold']),
        'F1': float(best['F1']),
        'Precision': float(best['Precision']),
        'Recall': float(best['Recall']),
        'Accuracy': float(best['Accuracy']),
    })

ensemble_grid = pd.DataFrame(rows).sort_values(['F1', 'Recall', 'Precision'], ascending=False).reset_index(drop=True)
ensemble_grid.to_csv(ARTIFACT_DIR / 'ensemble_weight_threshold_search.csv', index=False)

ens_best = ensemble_grid.iloc[0].to_dict()
print('Best ensemble:', ens_best)

best_w_bert = float(ens_best['w_bert'])
best_th = float(ens_best['best_threshold'])
test_prob_ens = best_w_bert * test_prob_bert + (1.0 - best_w_bert) * test_prob_lr
test_pred_ens = (test_prob_ens >= best_th).astype(int)

pd.DataFrame({'id': test_df['id'], 'target': test_pred_ens}).to_csv(
    ARTIFACT_DIR / 'submission_ensemble_soft_vote.csv',
    index=False,
)
pd.DataFrame({'id': test_df['id'], 'prob_disaster': test_prob_ens}).to_csv(
    ARTIFACT_DIR / 'ensemble_test_probabilities.csv',
    index=False,
)


Best ensemble: {'w_bert': 0.95, 'w_lr': 0.050000000000000044, 'best_threshold': 0.53, 'F1': 0.8103727714748784, 'Precision': 0.8389261744966443, 'Recall': 0.7836990595611285, 'Accuracy': 0.844103930712858}


In [ ]:
# 4) Class weight adjustment experiment (same split)
cls = np.array([0, 1])
weights = compute_class_weight(class_weight='balanced', classes=cls, y=y_tr)
print('class weights:', dict(zip(cls.tolist(), weights.tolist())))

va_prob_cw, va_true_cw, test_prob_cw, hist_cw = train_transformer_once(
    x_tr_bert,
    y_tr,
    x_va_bert,
    y_va,
    x_te_bert,
    class_weights=weights,
)

hist_cw.to_csv(ARTIFACT_DIR / 'transformer_class_weight_history.csv', index=False)

cw_curve, cw_best = threshold_sweep(va_true_cw, va_prob_cw)
cw_curve.to_csv(ARTIFACT_DIR / 'transformer_class_weight_threshold_curve.csv', index=False)

compare_df = pd.DataFrame([
    {
        'variant': 'BERTweet_default',
        'best_threshold': float(bert_best['threshold']),
        'F1': float(bert_best['F1']),
        'Precision': float(bert_best['Precision']),
        'Recall': float(bert_best['Recall']),
        'Accuracy': float(bert_best['Accuracy']),
    },
    {
        'variant': 'BERTweet_class_weight',
        'best_threshold': float(cw_best['threshold']),
        'F1': float(cw_best['F1']),
        'Precision': float(cw_best['Precision']),
        'Recall': float(cw_best['Recall']),
        'Accuracy': float(cw_best['Accuracy']),
    },
]).sort_values(['F1', 'Recall', 'Precision'], ascending=False).reset_index(drop=True)

compare_df.to_csv(ARTIFACT_DIR / 'class_weight_comparison.csv', index=False)
print(compare_df)


class weights: {0: 0.8701072774717309, 1: 1.1754798276537406}


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

{'epoch': 1, 'loss': 0.4904302862532278, 'F1': 0.7901668129938543, 'Precision': 0.8982035928143712, 'Recall': 0.7053291536050157, 'Accuracy': 0.8407728181212525}
{'epoch': 2, 'loss': 0.3608393403007946, 'F1': 0.8, 'Precision': 0.8169934640522876, 'Recall': 0.7836990595611285, 'Accuracy': 0.8334443704197202}
{'epoch': 3, 'loss': 0.28474412925363063, 'F1': 0.802610114192496, 'Precision': 0.8367346938775511, 'Recall': 0.7711598746081505, 'Accuracy': 0.8387741505662891}
                 variant  best_threshold        F1  Precision    Recall  \
0       BERTweet_default            0.41  0.809562   0.823339  0.796238   
1  BERTweet_class_weight            0.35  0.805423   0.819805  0.791536   

   Accuracy  
0  0.840773  
1  0.837442  


In [ ]:
# 5) Deeper error analysis: baseline vs transformer
use_cw = float(cw_best['F1']) > float(bert_best['F1'])
tr_name = 'BERTweet_class_weight' if use_cw else 'BERTweet_default'
tr_prob = va_prob_cw if use_cw else va_prob_bert
tr_th = float(cw_best['threshold']) if use_cw else float(bert_best['threshold'])

lr_th = float(lr_best['threshold'])

err = va[['id', 'text', 'target', 'keyword_clean', 'location', 'text_clean', 'text_transformer']].copy().reset_index(drop=True)
err['baseline_prob'] = va_prob_lr
err['transformer_prob'] = tr_prob

err['baseline_pred'] = (err['baseline_prob'] >= lr_th).astype(int)
err['transformer_pred'] = (err['transformer_prob'] >= tr_th).astype(int)

err['baseline_error'] = err['baseline_pred'] != err['target']
err['transformer_error'] = err['transformer_pred'] != err['target']

err['case_type'] = np.select(
    [
        (~err['baseline_error']) & (~err['transformer_error']),
        (err['baseline_error']) & (err['transformer_error']),
        (err['baseline_error']) & (~err['transformer_error']),
        (~err['baseline_error']) & (err['transformer_error']),
    ],
    [
        'both_correct',
        'both_wrong',
        'baseline_only_wrong',
        'transformer_only_wrong',
    ],
    default='unknown',
)

case_summary = err['case_type'].value_counts(dropna=False).rename_axis('case_type').reset_index(name='count')
case_summary['ratio'] = case_summary['count'] / len(err)
case_summary.to_csv(ARTIFACT_DIR / 'error_comparison_summary.csv', index=False)

err[err['case_type'] == 'baseline_only_wrong'].to_csv(ARTIFACT_DIR / 'error_cases_baseline_only_wrong.csv', index=False)
err[err['case_type'] == 'transformer_only_wrong'].to_csv(ARTIFACT_DIR / 'error_cases_transformer_only_wrong.csv', index=False)
err[err['case_type'] == 'both_wrong'].to_csv(ARTIFACT_DIR / 'error_cases_both_wrong.csv', index=False)

# FP/FN breakdown per model
for model_col, name in [('baseline_pred', 'baseline'), ('transformer_pred', 'transformer')]:
    fp = err[(err['target'] == 0) & (err[model_col] == 1)]
    fn = err[(err['target'] == 1) & (err[model_col] == 0)]
    fp.to_csv(ARTIFACT_DIR / f'{name}_false_positives_val.csv', index=False)
    fn.to_csv(ARTIFACT_DIR / f'{name}_false_negatives_val.csv', index=False)


def top_tokens(texts, n=40):
    c = Counter()
    for t in texts.fillna('').astype(str):
        toks = [tok for tok in t.split() if len(tok) > 2]
        c.update(toks)
    return pd.DataFrame(c.most_common(n), columns=['token', 'count'])


top_tokens(err.loc[err['case_type'] == 'baseline_only_wrong', 'text_clean'], n=40).to_csv(
    ARTIFACT_DIR / 'top_tokens_baseline_only_wrong.csv',
    index=False,
)
top_tokens(err.loc[err['case_type'] == 'transformer_only_wrong', 'text_clean'], n=40).to_csv(
    ARTIFACT_DIR / 'top_tokens_transformer_only_wrong.csv',
    index=False,
)

print('Transformer used in error analysis:', tr_name)
print(case_summary)


Transformer used in error analysis: BERTweet_default
                case_type  count     ratio
0            both_correct   1116  0.743504
1              both_wrong    163  0.108594
2     baseline_only_wrong    146  0.097268
3  transformer_only_wrong     76  0.050633


In [ ]:
# Final summary export
summary = {
    'split': {
        'seed': SEED,
        'val_size': VAL_SIZE,
        'train_size': int(len(tr)),
        'val_size_count': int(len(va)),
    },
    'baseline_lr': {
        'best_threshold': float(lr_best['threshold']),
        'F1': float(lr_best['F1']),
        'Precision': float(lr_best['Precision']),
        'Recall': float(lr_best['Recall']),
        'Accuracy': float(lr_best['Accuracy']),
    },
    'transformer_default': {
        'best_threshold': float(bert_best['threshold']),
        'F1': float(bert_best['F1']),
        'Precision': float(bert_best['Precision']),
        'Recall': float(bert_best['Recall']),
        'Accuracy': float(bert_best['Accuracy']),
    },
    'transformer_class_weight': {
        'best_threshold': float(cw_best['threshold']),
        'F1': float(cw_best['F1']),
        'Precision': float(cw_best['Precision']),
        'Recall': float(cw_best['Recall']),
        'Accuracy': float(cw_best['Accuracy']),
    },
    'chosen_transformer_for_error_analysis': tr_name,
    'ensemble_best': {
        'w_bert': float(ens_best['w_bert']),
        'w_lr': float(ens_best['w_lr']),
        'best_threshold': float(ens_best['best_threshold']),
        'F1': float(ens_best['F1']),
        'Precision': float(ens_best['Precision']),
        'Recall': float(ens_best['Recall']),
        'Accuracy': float(ens_best['Accuracy']),
    },
}

save_json(summary, ARTIFACT_DIR / 'step3_2_next_steps_summary.json')

print('Saved summary JSON to:', ARTIFACT_DIR / 'step3_2_next_steps_summary.json')
print('Done.')


Saved summary JSON to: /content/drive/MyDrive/CS544-Group17-Project/step3_2_next_steps_artifacts/step3_2_next_steps_summary.json
Done.


## Expected Outputs

Main files written to `step3_2_next_steps_artifacts/`:

- `baseline_threshold_curve.csv`
- `baseline_threshold_summary.json`
- `transformer_default_history.csv`
- `transformer_default_threshold_curve.csv`
- `transformer_class_weight_history.csv`
- `transformer_class_weight_threshold_curve.csv`
- `class_weight_comparison.csv`
- `ensemble_weight_threshold_search.csv`
- `submission_ensemble_soft_vote.csv`
- `error_comparison_summary.csv`
- `error_cases_baseline_only_wrong.csv`
- `error_cases_transformer_only_wrong.csv`
- `error_cases_both_wrong.csv`
- `baseline_false_positives_val.csv`
- `baseline_false_negatives_val.csv`
- `transformer_false_positives_val.csv`
- `transformer_false_negatives_val.csv`
- `step3_2_next_steps_summary.json`
